In [1]:
import pandas as pd
import numpy as np

In [2]:
url = 'https://raw.githubusercontent.com/campusx-official/100-days-of-machine-learning/refs/heads/main/day28-column-transformer/covid_toy.csv'
df = pd.read_csv(url)

In [3]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [4]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [5]:
df['city'].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [6]:
df['cough'].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

## Noraml Encoding 

In [7]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(df.iloc[:,0:5],df.iloc[:,-1],test_size=0.2)

x_train

In [8]:
y_train.head(5) 

73    Yes
26    Yes
31     No
5     Yes
8      No
Name: has_covid, dtype: object

In Pandas, a single column is called a Series.
Pandas does not display a Series as a table.

**SimpleImputer** is a tool in scikit-learn used to handle missing values (NaNs) in a dataset by replacing them with a specific statistical value.

**How it works**
1. strategy='mean' (Default): Replaces missing numbers with the average of that column.

2. strategy='median': Replaces missing numbers with the middle value (best if there are extreme outliers).

3. strategy='most_frequent': Replaces missing data with the most common value (used for text/categorical data).


**Why use it?** Machine learning models cannot process missing data. SimpleImputer fixes this automatically without dropping rows, preserving your data size.


In [9]:
# to handle null values 
from sklearn.impute import SimpleImputer 

imputer = SimpleImputer() 
x_train_fever = imputer.fit_transform(x_train[['fever']])

x_test_fever = imputer.transform(x_test[['fever']])

In [10]:
x_train_fever.shape

(80, 1)

fit() --> learn from dataset                                                                                                     
transform() -->Apply on data whaterver learn

Whenever pass a single column into Scikit-Learn tools (SimpleImputer, OneHotEncoder, etc) always use [[]] to keep it in a table format and avoid errors.

In [11]:
# OrdinalEncoding ---> cough
from sklearn.preprocessing import OrdinalEncoder
Or = OrdinalEncoder( categories=[['Mild','Strong']])

x_train_cough = Or.fit_transform(x_train[['cough']])
x_test_cough = Or.transform(x_test[['cough']])

In [12]:
x_train_cough.shape

(80, 1)

In [13]:
# OHE --> gender,city
from sklearn.preprocessing import OneHotEncoder
ohn = OneHotEncoder(drop='first', sparse_output=False)

x_train_gender = ohn.fit_transform(x_train[['gender','city']])

x_test_gender = ohn.transform(x_test[['gender','city']])


In [14]:
x_train_gender.shape

(80, 4)

In [15]:
#Etracting Age
x_train_age = x_train.iloc[:,0:1]

x_test_age = x_test.iloc[:,0:1]

In [16]:
x_train_age.shape

(80, 1)

In [17]:
x_train_tranformed = np.concatenate((x_train_age,x_train_gender,x_train_cough,x_train_fever),axis=1)

x_test_tranformed = np.concatenate((x_test_age,x_test_gender,x_test_cough,x_test_fever),axis=1)

In [18]:
x_train_tranformed.shape

(80, 7)

## Column Tranformer 

In [20]:
from sklearn.compose import ColumnTransformer

In [36]:
transform = ColumnTransformer(
    transformers=[
        ('tnf1',SimpleImputer(),['fever']),
        ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
        ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
    ],remainder='passthrough')

In [38]:
transform.fit_transform(x_train).shape

(80, 7)

In [40]:
transform.transform(x_test).shape

(20, 7)